In [ ]:
import jax
import jax.numpy as jnp
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

In [ ]:
jax.devices()

In [ ]:
KEY = jax.random.PRNGKey(12)


def random(*shape):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    return jax.random.normal(subkey, shape)

In [ ]:
def get_synced(systemA, systemB, sync_over):
    def foo(stateA, stateB, intensivity):
        s1 = systemA(stateA)
        s2 = systemB(stateB)
        for sync in sync_over:
            path1, path2, w = sync
            s_v = s1[path1] * w + s2[path2] * (1 - w)
            b, a = 1 - intensivity, intensivity
            s1[path1] = s_v * a + s1[path1] * b
            s2[path2] = s_v * a + s2[path2] * b
        return s1, s2

    return foo

In [ ]:
def lenght(x):
    return jnp.sum(x**2) ** 0.5


def normalize(x):
    L = lenght(x)
    return jax.lax.cond(L == 0, lambda x: jnp.zeros_like(x), lambda x: x / L, x)


def get_transformation(dt):
    def transform_state(state):
        x, y, z = state["x"], state["y"], state["z"]
        m = {k: lenght(v) for k, v in state.items()}

        new_state = {
            "x": x + dt * ((z @ x) * y - (y @ x) * z),
            "y": y + dt * ((x @ y) * z - (z @ y) * x),
            "z": z + dt * ((y @ z) * x - (x @ z) * y),
        }

        return jax.tree_util.tree_map(lambda a, b: normalize(a) * b, new_state, m)

    return transform_state

In [ ]:
DIM = 10

s1i = {k: random(DIM) for k in ("x", "y", "z")}

s2i = {k: random(DIM) for k in ("x", "y", "z")}

In [ ]:
system1 = system2 = get_transformation(0.01)

In [ ]:
s = get_synced(system1, system2, [["x", "x", 0.9]])
s1, s2 = s1i, s2i
SYNC_START = 3000
SYNC_END = 10000


# 2. Define the scan body for the 100 macro-steps (1000 total steps / 10)
def scan_body(carry, _):
    s1, s2, i = carry
    s1_next, s2_next = s(s1, s2, (SYNC_START < i) * (i < SYNC_END))

    # Compute metrics for all pairs in parallel/compiled form
    metrics = {
        (k1, k2): jnp.mean((v1 - v2) ** 2)
        for k1, v1 in s1_next.items()
        for k2, v2 in s2_next.items()
    }

    return (s1_next, s2_next, i + 1), metrics


# 3. JIT-compile the entire simulation loop
@jax.jit
def run_simulation(s1_init, s2_init):
    final_state, history_stacked = jax.lax.scan(
        scan_body, (s1_init, s2_init, 0), None, length=20000
    )
    return final_state, history_stacked


# --- Execution ---
s1, s2 = s1i, s2i
final_state, H_stacked = run_simulation(s1, s2)

In [ ]:
for k, v in H_stacked.items():
    plt.title(str(k))
    plt.plot(v)
    plt.axvspan(SYNC_START, SYNC_END, color="yellow")
    plt.show()